# Analyze Temporal Feature Signal

Диагностика признаков из `adausdt_minute_log_return_temporal_dataset.parquet`:

- baseline модель на time split;
- permutation importance;
- mutual information;
- autocorrelation target/features;
- SHAP, если установлен пакет `shap`.

Датасет большой, поэтому анализ работает на равномерном sample по parquet row groups. Это быстрее и достаточно для первого feature screening.

In [3]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 160)

try:
    display
except NameError:
    display = print

## Settings

In [4]:
DATASET_PATH = Path("model_artifacts/adausdt_minute_log_return_temporal_dataset.parquet")
REPORT_PATH = Path("model_artifacts/temporal_feature_signal_report.json")

TARGET_COL = "log_return_1m"
NON_FEATURE_COLS = {"date", "timestamp", "log_return_1m", "return_1m"}

# Keep the first pass intentionally compact. Increase these after the first read.
SAMPLE_ROWS = 250_000
MODEL_FEATURE_LIMIT = 350
PERMUTATION_FEATURE_LIMIT = 80
MI_FEATURE_LIMIT = 250
SHAP_BACKGROUND_ROWS = 2_000
SHAP_EXPLAIN_ROWS = 5_000

RANDOM_STATE = 42
TRAIN_FRACTION = 0.8
ACF_LAGS = [1, 2, 3, 5, 10, 15, 30, 60, 120]

assert DATASET_PATH.exists(), DATASET_PATH

## Schema And Column Selection

In [5]:
pf = pq.ParquetFile(DATASET_PATH)
all_columns = pf.schema_arrow.names
feature_cols = [col for col in all_columns if col not in NON_FEATURE_COLS]

print("rows", pf.metadata.num_rows)
print("row_groups", pf.num_row_groups)
print("columns", len(all_columns))
print("feature_columns", len(feature_cols))

display(pd.Series(feature_cols).head(30).to_frame("feature"))

rows 3157859
row_groups 4
columns 404
feature_columns 400


,feature
0,aggression_features__V_buy_quote
1,aggression_features__V_sell_quote
2,aggression_features__delta_base_norm
3,aggression_features__delta_quote_norm
4,btc_features__btc_log_return
5,btc_features__btc_rolling_volatility
6,btc_features__btc_zscore
7,intraminute_dynamics__F_concentration
8,intraminute_dynamics__RV
9,intraminute_dynamics__pressure_segment_3


Для модели сначала ограничим число фичей: берём все фичи, но если их слишком много, ранжируем по variance на маленьком sample и оставляем самые вариативные. Это не финальный отбор, а способ сделать первый анализ быстрым.

In [6]:
def sample_parquet_rows(columns, sample_rows=SAMPLE_ROWS, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    per_group = max(1, math.ceil(sample_rows / pf.num_row_groups))
    parts = []
    for group_idx in range(pf.num_row_groups):
        table = pf.read_row_group(group_idx, columns=columns)
        part = table.to_pandas()
        if len(part) > per_group:
            idx = np.sort(rng.choice(len(part), size=per_group, replace=False))
            part = part.iloc[idx]
        parts.append(part)
    out = pd.concat(parts, ignore_index=True)
    if "timestamp" in out.columns:
        out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True)
        out = out.sort_values("timestamp").reset_index(drop=True)
    if len(out) > sample_rows:
        keep = np.linspace(0, len(out) - 1, sample_rows).round().astype(int)
        out = out.iloc[keep].reset_index(drop=True)
    return out


small_cols = ["timestamp", TARGET_COL] + feature_cols
small_sample_rows = min(80_000, SAMPLE_ROWS)
small = sample_parquet_rows(small_cols, sample_rows=small_sample_rows)
variances = small[feature_cols].var(numeric_only=True).sort_values(ascending=False)
model_feature_cols = variances.head(min(MODEL_FEATURE_LIMIT, len(variances))).index.tolist()

print("small sample", small.shape)
print("selected model features", len(model_feature_cols))
display(variances.head(30).to_frame("variance"))

small sample (72131, 402)
selected model features 350


,variance
price_pressure__F_asymmetry__roll_std_30,3.727453e+23
price_pressure__F_asymmetry__roll_std_15,2.021167e+23
price_pressure__F_asymmetry__lag_3,5.835260e+22
price_pressure__F_asymmetry__lag_5,4.081229e+22
price_pressure__F_asymmetry__lag_10,3.305857e+22
price_pressure__F_asymmetry__roll_mean_15,1.462419e+22
price_pressure__F_asymmetry__roll_mean_30,1.312181e+22
price_pressure__F_asymmetry__roll_std_5,1.122993e+22
price_pressure__F_asymmetry__ewm_mean_30,4.776429e+21
price_pressure__F_asymmetry__lag_1,4.512992e+21


## Load Analysis Sample

In [7]:
analysis_cols = ["date", "timestamp", TARGET_COL] + model_feature_cols
data = sample_parquet_rows(analysis_cols, sample_rows=SAMPLE_ROWS)
data = data.dropna(subset=[TARGET_COL]).reset_index(drop=True)

print(data.shape)
print(data["timestamp"].min(), data["timestamp"].max())
display(data[["date", "timestamp", TARGET_COL]].head())
display(data[["date", "timestamp", TARGET_COL]].tail())

(199631, 353)
2020-02-01 01:01:00+00:00 2026-02-01 23:59:00+00:00


,date,timestamp,log_return_1m
0,2020-02-01,2020-02-01 01:01:00+00:00,0.000000
1,2020-02-01,2020-02-01 01:04:00+00:00,0.001839
2,2020-02-01,2020-02-01 01:14:00+00:00,0.000000
3,2020-02-01,2020-02-01 01:20:00+00:00,0.002018
4,2020-02-01,2020-02-01 01:44:00+00:00,-0.000367


,date,timestamp,log_return_1m
199626,2026-02-01,2026-02-01 23:55:00+00:00,0.000700
199627,2026-02-01,2026-02-01 23:56:00+00:00,0.001399
199628,2026-02-01,2026-02-01 23:57:00+00:00,-0.002099
199629,2026-02-01,2026-02-01 23:58:00+00:00,0.000350
199630,2026-02-01,2026-02-01 23:59:00+00:00,0.001050


## Baseline Model

In [8]:
split_idx = int(len(data) * TRAIN_FRACTION)
train = data.iloc[:split_idx].copy()
test = data.iloc[split_idx:].copy()

X_train = train[model_feature_cols]
y_train = train[TARGET_COL]
X_test = test[model_feature_cols]
y_test = test[TARGET_COL]

model = make_pipeline(
    SimpleImputer(strategy="median"),
    HistGradientBoostingRegressor(
        max_iter=250,
        learning_rate=0.04,
        max_leaf_nodes=31,
        l2_regularization=0.0,
        random_state=RANDOM_STATE,
    ),
)
model.fit(X_train, y_train)
pred = model.predict(X_test)

baseline_metrics = {
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "train_end": str(train["timestamp"].max()),
    "test_start": str(test["timestamp"].min()),
    "mae": float(mean_absolute_error(y_test, pred)),
    "rmse": float(mean_squared_error(y_test, pred) ** 0.5),
    "r2": float(r2_score(y_test, pred)),
    "directional_accuracy": float((np.sign(pred) == np.sign(y_test)).mean()),
}
baseline_metrics

{'train_rows': 159704,
 'test_rows': 39927,
 'train_end': '2025-03-06 20:47:00+00:00',
 'test_start': '2025-03-06 21:09:00+00:00',
 'mae': 0.000798354087354718,
 'rmse': 0.001296106015318484,
 'r2': 0.009901866546654592,
 'directional_accuracy': 0.44248253061837856}

## Permutation Importance

In [9]:
perm_feature_cols = model_feature_cols[:PERMUTATION_FEATURE_LIMIT]
perm_rows = min(50_000, len(X_test))
X_perm = X_test[perm_feature_cols].tail(perm_rows)
y_perm = y_test.tail(perm_rows)

# Refit a smaller model on the same feature subset to make permutation faster and more interpretable.
perm_model = make_pipeline(
    SimpleImputer(strategy="median"),
    HistGradientBoostingRegressor(max_iter=180, learning_rate=0.05, max_leaf_nodes=31, random_state=RANDOM_STATE),
)
perm_model.fit(X_train[perm_feature_cols], y_train)

perm = permutation_importance(
    perm_model,
    X_perm,
    y_perm,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="neg_mean_squared_error",
    n_jobs=1,
)
permutation_importance_df = (
    pd.DataFrame({
        "feature": perm_feature_cols,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
display(permutation_importance_df.head(50))

,feature,importance_mean,importance_std
0,price_pressure__F_PI__roll_std_15,8.842112e-09,2.905101e-10
1,intraminute_dynamics__pressure_segment_3__roll_max_5,5.306659e-09,2.897429e-10
2,price_pressure__F_PI__roll_mean_15,2.457185e-09,7.909453e-10
3,intraminute_dynamics__pressure_segment_3__roll_mean_5,2.350986e-09,0.000000e+00
4,trade_distribution__N_eff_buy__roll_sum_60,2.204466e-09,6.153216e-11
5,price_pressure__F_PI__ewm_mean_5,1.865799e-09,1.468753e-09
6,price_pressure__F_asymmetry__ewm_mean_5,1.678876e-09,3.197012e-10
7,intraminute_dynamics__F_concentration,1.600583e-09,6.097073e-10
8,aggression_features__V_buy_quote__lag_10,1.447987e-09,0.000000e+00
9,intraminute_dynamics__pressure_total__roll_max_5,1.197012e-09,3.341136e-11


## Mutual Information

In [10]:
mi_feature_cols = model_feature_cols[:MI_FEATURE_LIMIT]
mi_rows = min(80_000, len(data))
mi_data = data.tail(mi_rows).copy()

mi_imputer = SimpleImputer(strategy="median")
X_mi = mi_imputer.fit_transform(mi_data[mi_feature_cols])
y_mi = mi_data[TARGET_COL].to_numpy()

mi_values = mutual_info_regression(X_mi, y_mi, random_state=RANDOM_STATE, n_neighbors=5)
mutual_info_df = (
    pd.DataFrame({"feature": mi_feature_cols, "mutual_information": mi_values})
    .sort_values("mutual_information", ascending=False)
    .reset_index(drop=True)
)
display(mutual_info_df.head(50))

,feature,mutual_information
0,intraminute_segments__VWAP_segment_3,2.937079
1,intraminute_segments__VWAP_minute,2.884121
2,intraminute_segments__VWAP_segment_2,2.872194
3,intraminute_segments__VWAP_segment_1,2.811921
4,intraminute_dynamics__pressure_total__roll_max_30,0.190977
5,intraminute_dynamics__pressure_segment_3__roll_max_30,0.188005
6,trade_distribution__N_eff_buy__roll_sum_60,0.170182
7,trade_distribution__N_eff_buy__roll_mean_60,0.170182
8,trade_distribution__N_eff_sell__roll_sum_60,0.165175
9,trade_distribution__N_eff_sell__roll_mean_60,0.165175


## Autocorrelation

In [11]:
# Target autocorrelation on the full target column, narrow read.
target_only = pd.read_parquet(DATASET_PATH, columns=["timestamp", TARGET_COL])
target_only["timestamp"] = pd.to_datetime(target_only["timestamp"], utc=True)
target_only = target_only.sort_values("timestamp").reset_index(drop=True)

target_acf_df = pd.DataFrame(
    [{"lag": lag, "target_autocorrelation": float(target_only[TARGET_COL].autocorr(lag=lag))} for lag in ACF_LAGS]
)
display(target_acf_df)

# Feature autocorrelation for union of top permutation and MI features.
top_features = list(dict.fromkeys(
    permutation_importance_df.head(25)["feature"].tolist()
    + mutual_info_df.head(25)["feature"].tolist()
))
feature_acf_rows = []
for col in top_features:
    for lag in [1, 5, 15, 60]:
        feature_acf_rows.append({"feature": col, "lag": lag, "autocorrelation": float(data[col].autocorr(lag=lag))})
feature_acf_df = pd.DataFrame(feature_acf_rows).sort_values(["lag", "autocorrelation"], ascending=[True, False])
display(feature_acf_df.head(80))

,lag,target_autocorrelation
0,1,0.007905
1,2,-0.014879
2,3,-0.000492
3,5,-0.001258
4,10,0.006179
5,15,0.009430
6,30,0.005731
7,60,0.006445
8,120,0.004293


,feature,lag,autocorrelation
104,intraminute_segments__VWAP_minute,1,0.999858
180,sinthetic_data__sin_weekday,1,0.999806
112,intraminute_segments__VWAP_segment_1,1,0.999795
108,intraminute_segments__VWAP_segment_2,1,0.999719
100,intraminute_segments__VWAP_segment_3,1,0.999691
124,trade_distribution__N_eff_buy__roll_mean_60,1,0.964736
16,trade_distribution__N_eff_buy__roll_sum_60,1,0.964736
128,trade_distribution__N_eff_sell__roll_sum_60,1,0.964063
132,trade_distribution__N_eff_sell__roll_mean_60,1,0.964063
60,trade_distribution__N_eff_buy__roll_sum_30,1,0.923418


## SHAP

Если `shap` не установлен, установи его в окружение и перезапусти этот блок:

```powershell
python -m pip install shap
```

In [12]:
shap_importance_df = pd.DataFrame(columns=["feature", "mean_abs_shap"])

try:
    import shap

    shap_feature_cols = perm_feature_cols
    shap_model = ExtraTreesRegressor(
        n_estimators=120,
        min_samples_leaf=30,
        max_features=0.7,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    shap_imputer = SimpleImputer(strategy="median")
    X_shap_train = shap_imputer.fit_transform(X_train[shap_feature_cols].tail(min(120_000, len(X_train))))
    y_shap_train = y_train.tail(min(120_000, len(y_train))).to_numpy()
    shap_model.fit(X_shap_train, y_shap_train)

    X_explain = shap_imputer.transform(X_test[shap_feature_cols].tail(min(SHAP_EXPLAIN_ROWS, len(X_test))))
    explainer = shap.TreeExplainer(shap_model)
    shap_values = explainer.shap_values(X_explain, check_additivity=False)
    shap_importance_df = (
        pd.DataFrame({
            "feature": shap_feature_cols,
            "mean_abs_shap": np.abs(shap_values).mean(axis=0),
        })
        .sort_values("mean_abs_shap", ascending=False)
        .reset_index(drop=True)
    )
    display(shap_importance_df.head(50))
except ModuleNotFoundError as exc:
    print("SHAP skipped:", exc)
    print("Install with: python -m pip install shap")

c:\Users\Пользователь\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,feature,mean_abs_shap
0,intraminute_dynamics__pressure_segment_3__roll_std_5,0.000005
1,intraminute_dynamics__pressure_total__roll_max_5,0.000004
2,price_pressure__PI_total__roll_std_15,0.000004
3,aggression_features__V_sell_quote,0.000004
4,intraminute_dynamics__pressure_total__roll_max_15,0.000004
5,intraminute_dynamics__pressure_segment_3__roll_max_5,0.000004
6,intraminute_dynamics__pressure_total__roll_max_30,0.000004
7,price_pressure__PI_total__roll_std_5,0.000004
8,intraminute_dynamics__pressure_total__roll_std_5,0.000004
9,aggression_features__V_sell_quote__lag_2,0.000004


## Combined Ranking And Save

In [13]:
combined = pd.DataFrame({"feature": sorted(set(
    permutation_importance_df["feature"].tolist()
    + mutual_info_df["feature"].tolist()
    + shap_importance_df["feature"].tolist()
))})
combined = combined.merge(permutation_importance_df, on="feature", how="left")
combined = combined.merge(mutual_info_df, on="feature", how="left")
combined = combined.merge(shap_importance_df, on="feature", how="left")

for metric in ["importance_mean", "mutual_information", "mean_abs_shap"]:
    if metric in combined.columns:
        rank_col = f"{metric}_rank"
        combined[rank_col] = combined[metric].rank(ascending=False, method="average", na_option="bottom")

rank_cols = [col for col in combined.columns if col.endswith("_rank")]
combined["average_rank"] = combined[rank_cols].mean(axis=1)
combined_ranking_df = combined.sort_values("average_rank").reset_index(drop=True)
display(combined_ranking_df.head(80))

out_dir = Path("model_artifacts/feature_signal")
out_dir.mkdir(parents=True, exist_ok=True)
permutation_importance_df.to_csv(out_dir / "permutation_importance.csv", index=False)
mutual_info_df.to_csv(out_dir / "mutual_information.csv", index=False)
target_acf_df.to_csv(out_dir / "target_autocorrelation.csv", index=False)
feature_acf_df.to_csv(out_dir / "feature_autocorrelation.csv", index=False)
shap_importance_df.to_csv(out_dir / "shap_importance.csv", index=False)
combined_ranking_df.to_csv(out_dir / "combined_feature_ranking.csv", index=False)

report = {
    "dataset": str(DATASET_PATH),
    "sample_rows": int(len(data)),
    "model_feature_count": int(len(model_feature_cols)),
    "baseline_metrics": baseline_metrics,
    "top_permutation": permutation_importance_df.head(30).to_dict(orient="records"),
    "top_mutual_information": mutual_info_df.head(30).to_dict(orient="records"),
    "target_autocorrelation": target_acf_df.to_dict(orient="records"),
    "top_shap": shap_importance_df.head(30).to_dict(orient="records"),
    "output_dir": str(out_dir),
}
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"saved report: {REPORT_PATH}")
print(f"saved csv files: {out_dir}")

,feature,importance_mean,importance_std,mutual_information,mean_abs_shap,importance_mean_rank,mutual_information_rank,mean_abs_shap_rank,average_rank
0,trade_distribution__N_eff_buy__roll_sum_60,2.204466e-09,6.153216e-11,0.170182,2.862185e-06,5.0,7.0,19.0,10.333333
1,intraminute_dynamics__pressure_total__roll_std_30,3.937610e-10,0.000000e+00,0.161785,3.280355e-06,22.0,13.0,12.0,15.666667
2,intraminute_dynamics__pressure_total__roll_max_30,0.000000e+00,0.000000e+00,0.190977,4.076199e-06,46.0,5.0,7.0,19.333333
3,intraminute_dynamics__pressure_total__roll_max_5,1.197012e-09,3.341136e-11,0.099504,4.483001e-06,10.0,56.0,2.0,22.666667
4,intraminute_dynamics__pressure_segment_3__roll_max_30,0.000000e+00,0.000000e+00,0.188005,2.955144e-06,46.0,6.0,17.0,23.000000
5,intraminute_dynamics__pressure_segment_3__roll_max_5,5.306659e-09,2.897429e-10,0.094422,4.222035e-06,2.0,62.0,6.0,23.333333
6,trade_distribution__N_eff_buy__roll_sum_30,6.282463e-10,0.000000e+00,0.161292,1.998840e-06,16.0,15.0,39.0,23.333333
7,price_pressure__PI_total__roll_std_30,0.000000e+00,0.000000e+00,0.161787,3.175956e-06,46.0,12.0,13.0,23.666667
8,trade_distribution__N_eff_sell__roll_sum_15,1.799710e-10,0.000000e+00,0.141922,2.787115e-06,29.0,22.0,21.0,24.000000
9,price_pressure__PI_total__roll_std_5,4.274766e-10,1.582750e-11,0.109790,3.890917e-06,20.0,46.0,8.0,24.666667


saved report: model_artifacts\temporal_feature_signal_report.json
saved csv files: model_artifacts\feature_signal
